In [1]:
!ls /kaggle/input

datasets


In [2]:
!ls /kaggle/input/datasets

prabhuingole


In [3]:
!ls /kaggle/input/datasets/prabhuingole

wastesegregationk1


In [4]:
!ls /kaggle/input/datasets/prabhuingole/wastesegregationk1

TACO-master


In [5]:
!ls /kaggle/input/datasets/prabhuingole/wastesegregationk1/TACO-master

data  demo.ipynb  detector  download.py  LICENSE  README.md  requirements.txt


In [6]:
!ls /kaggle/input/datasets/prabhuingole/wastesegregationk1/TACO-master/data

all_image_urls.csv  annotations.json  annotations_unofficial.json


In [7]:
!cp -r /kaggle/input/datasets/prabhuingole/wastesegregationk1/TACO-master .

In [8]:
!ls



TACO-master


In [9]:
!python TACO-master/download.py

Note. If for any reason the connection is broken. Just call me again and I will start where I left.
Traceback (most recent call last):
  File "/kaggle/working/TACO-master/download.py", line 23, in <module>
    with open(args.dataset_path, 'r') as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: './data/annotations.json'


In [10]:
!git clone https://github.com/pedropro/TACO.git

Cloning into 'TACO'...
remote: Enumerating objects: 740, done.
remote: Counting objects: 100% (435/435), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 740 (delta 416), reused 380 (delta 380), pack-reused 305 (from 1)
Receiving objects: 100% (740/740), 97.48 MiB | 34.09 MiB/s, done.
Resolving deltas: 100% (499/499), done.


In [11]:
!ls /kaggle/working/TACO/data

all_image_urls.csv  annotations.json  annotations_unofficial.json


In [12]:
!python /kaggle/working/TACO/download.py \
--dataset_path /kaggle/working/TACO/data/annotations.json \
--output_dir /kaggle/working/taco_dataset

usage: download.py [-h] [--dataset_path DATASET_PATH]
download.py: error: unrecognized arguments: --output_dir /kaggle/working/taco_dataset


In [13]:
!python /kaggle/working/TACO/download.py \
--dataset_path /kaggle/working/TACO/data/annotations.json


Note. If for any reason the connection is broken. Just call me again and I will start where I left.
Finished [=============================.] - 1499/1500


In [14]:
!ls /kaggle/working/TACO/data | head

all_image_urls.csv
annotations.json
annotations_unofficial.json
batch_1
batch_10
batch_11
batch_12
batch_13
batch_14
batch_15


In [15]:
import json

with open('/kaggle/working/TACO/data/annotations.json') as f:
    data = json.load(f)

print("Number of images:", len(data["images"]))
print("Number of annotations:", len(data["annotations"]))
print("Number of categories:", len(data["categories"]))

Number of images: 1500
Number of annotations: 4784
Number of categories: 60


In [16]:
super_categories = set()

for cat in data["categories"]:
    super_categories.add(cat["supercategory"])

print(super_categories)
print("Total:", len(super_categories))

{'Lid', 'Glass jar', 'Paper', 'Aluminium foil', 'Rope & strings', 'Plastic bag & wrapper', 'Bottle cap', 'Plastic glooves', 'Plastic utensils', 'Other plastic', 'Battery', 'Paper bag', 'Unlabeled litter', 'Bottle', 'Blister pack', 'Broken glass', 'Squeezable tube', 'Food waste', 'Cigarette', 'Scrap metal', 'Styrofoam piece', 'Plastic container', 'Can', 'Pop tab', 'Carton', 'Straw', 'Shoe', 'Cup'}
Total: 28


In [17]:
!ls /kaggle/working/TACO/data/batch_1 | head

000000.jpg
000001.jpg
000003.jpg
000004.jpg
000005.jpg
000006.jpg
000007.jpg
000008.jpg
000010.jpg
000011.jpg


In [18]:
!ls /kaggle/working/TACO/data/batch_2 | head

000000.JPG
000001.JPG
000003.JPG
000005.JPG
000006.JPG
000007.JPG
000008.JPG
000009.JPG
000010.JPG
000012.JPG


In [19]:
import json
import os
from collections import defaultdict

In [20]:
annotation_path = "/kaggle/working/TACO/data/annotations.json"

with open(annotation_path) as f:
    data = json.load(f)

print("Images:", len(data["images"]))
print("Annotations:", len(data["annotations"]))
print("Categories:", len(data["categories"]))

Images: 1500
Annotations: 4784
Categories: 60


In [21]:
image_id_to_info = {}

for img in data["images"]:
    image_id_to_info[img["id"]] = img

In [22]:
image_annotations = defaultdict(list)

for ann in data["annotations"]:
    image_annotations[ann["image_id"]].append(ann)

In [23]:
label_dir = "/kaggle/working/yolo_labels"
os.makedirs(label_dir, exist_ok=True)

In [24]:
for image_id, anns in image_annotations.items():

    img_info = image_id_to_info[image_id]

    file_name = img_info["file_name"]
    width = img_info["width"]
    height = img_info["height"]

    label_file = file_name.split("/")[-1].replace(".jpg", ".txt")

    label_path = os.path.join(label_dir, label_file)

    with open(label_path, "w") as f:

        for ann in anns:

            class_id = ann["category_id"]

            x, y, w, h = ann["bbox"]

            x_center = (x + w / 2) / width
            y_center = (y + h / 2) / height
            w = w / width
            h = h / height

            f.write(f"{class_id} {x_center} {y_center} {w} {h}\n")

In [25]:
!ls /kaggle/working/yolo_labels | head

000000.JPG
000000.txt
000001.JPG
000001.txt
000002.JPG
000002.txt
000003.JPG
000003.txt
000004.JPG
000004.txt


In [26]:
!cat /kaggle/working/yolo_labels/000000.txt

43 0.6358762254901961 0.5183823529411765 0.046875 0.07107843137254902
31 0.44515931372549017 0.7173202614379085 0.025735294117647058 0.035130718954248366
36 0.375 0.7412173202614379 0.044730392156862746 0.0298202614379085


In [27]:
supercats = sorted(set(cat["supercategory"] for cat in data["categories"]))

supercat_to_id = {name:i for i,name in enumerate(supercats)}

print(supercat_to_id)

{'Aluminium foil': 0, 'Battery': 1, 'Blister pack': 2, 'Bottle': 3, 'Bottle cap': 4, 'Broken glass': 5, 'Can': 6, 'Carton': 7, 'Cigarette': 8, 'Cup': 9, 'Food waste': 10, 'Glass jar': 11, 'Lid': 12, 'Other plastic': 13, 'Paper': 14, 'Paper bag': 15, 'Plastic bag & wrapper': 16, 'Plastic container': 17, 'Plastic glooves': 18, 'Plastic utensils': 19, 'Pop tab': 20, 'Rope & strings': 21, 'Scrap metal': 22, 'Shoe': 23, 'Squeezable tube': 24, 'Straw': 25, 'Styrofoam piece': 26, 'Unlabeled litter': 27}


In [28]:
cat_to_super = {}

for cat in data["categories"]:
    
    cat_id = cat["id"]
    supercat = cat["supercategory"]
    
    cat_to_super[cat_id] = supercat_to_id[supercat]

In [29]:
import os

label_dir = "/kaggle/working/yolo_labels"

for file in os.listdir(label_dir):

    if file.endswith(".txt"):
        
        path = os.path.join(label_dir, file)
        
        new_lines = []
        
        with open(path) as f:
            lines = f.readlines()
            
        for line in lines:
            
            parts = line.strip().split()
            
            old_class = int(parts[0])
            
            new_class = cat_to_super[old_class]
            
            parts[0] = str(new_class)
            
            new_lines.append(" ".join(parts))
        
        with open(path, "w") as f:
            f.write("\n".join(new_lines))

In [30]:
cat 000000.txt

SyntaxError: invalid decimal literal (134919945.py, line 1)

In [ ]:
!cat 000000.txt

In [31]:
!ls | head

TACO
TACO-master
yolo_labels


In [32]:
!ls 

TACO  TACO-master  yolo_labels


In [33]:
import os
import shutil

image_root = "/kaggle/working/TACO/data"
label_root = "/kaggle/working/yolo_labels"

for batch in os.listdir(image_root):

    if batch.startswith("batch"):

        batch_path = os.path.join(image_root, batch)

        for img in os.listdir(batch_path):

            if img.lower().endswith(".jpg") or img.lower().endswith(".JPG"):

                label = img.replace(".JPG", ".txt").replace(".jpg", ".txt")

                label_path = os.path.join(label_root, label)

                if os.path.exists(label_path):

                    shutil.move(label_path, os.path.join(batch_path, label))

print("Labels moved successfully.")

Labels moved successfully.


In [34]:
!ls yolo_labels | wc -l

239


In [35]:
len(set([ann["image_id"] for ann in data["annotations"]]))

1500

In [36]:
import os

label_dir = "/kaggle/working/yolo_labels"
os.makedirs(label_dir, exist_ok=True)

for image_id, anns in image_annotations.items():

    img_info = image_id_to_info[image_id]

    file_name = img_info["file_name"]   # batch_1/000000.jpg

    width = img_info["width"]
    height = img_info["height"]

    batch, img = file_name.split("/")

    label_name = batch + "_" + img.replace(".jpg",".txt").replace(".JPG",".txt")

    label_path = os.path.join(label_dir, label_name)

    with open(label_path, "w") as f:

        for ann in anns:

            class_id = ann["category_id"]

            x, y, w, h = ann["bbox"]

            x_center = (x + w/2) / width
            y_center = (y + h/2) / height
            w = w / width
            h = h / height

            f.write(f"{class_id} {x_center} {y_center} {w} {h}\n")

In [37]:
!ls yolo_labels | wc -l

1739


In [38]:
!rm yolo_labels/*.txt

In [39]:
!ls yolo_labels | wc -l


239


In [40]:
!ls yolo_labels | wc -l

239


In [41]:
!rm yolo_labels/*.txt

rm: cannot remove 'yolo_labels/*.txt': No such file or directory


In [42]:
!rm -rf yolo_labels

In [43]:
!ls yolo_labels | wc -l

ls: cannot access 'yolo_labels': No such file or directory
0


In [44]:
!mkdir yolo_labels

In [45]:
!ls yolo_labels | wc -l

0


In [46]:
import os

label_dir = "/kaggle/working/yolo_labels"
os.makedirs(label_dir, exist_ok=True)

for image_id, anns in image_annotations.items():

    img_info = image_id_to_info[image_id]

    file_name = img_info["file_name"]   # batch_1/000000.jpg

    width = img_info["width"]
    height = img_info["height"]

    batch, img = file_name.split("/")

    label_name = batch + "_" + img.replace(".jpg",".txt").replace(".JPG",".txt")

    label_path = os.path.join(label_dir, label_name)

    with open(label_path, "w") as f:

        for ann in anns:

            class_id = ann["category_id"]

            x, y, w, h = ann["bbox"]

            x_center = (x + w/2) / width
            y_center = (y + h/2) / height
            w = w / width
            h = h / height

            f.write(f"{class_id} {x_center} {y_center} {w} {h}\n")

In [47]:
!ls yolo_labels | wc -l

1500


In [48]:
import os

label_dir = "/kaggle/working/yolo_labels"

for file in os.listdir(label_dir):

    if file.endswith(".txt"):

        path = os.path.join(label_dir,file)

        new_lines = []

        with open(path) as f:
            lines = f.readlines()

        for line in lines:

            parts = line.strip().split()

            old_class = int(parts[0])

            new_class = cat_to_super[old_class]

            parts[0] = str(new_class)

            new_lines.append(" ".join(parts))

        with open(path,"w") as f:
            f.write("\n".join(new_lines))

In [49]:
!pip install ultralytics
from ultralytics import YOLO
YOLO("yolov8s.pt")
import yaml

data_yaml = {
    "path": "/kaggle/working/TACO/data",  # dataset root
    "train": ".",                         # all images inside batches
    "val": ".",                           # using same for now
    "nc": 28,                             # number of classes
    "names": list(supercat_to_id.keys())  # supercategory names
}

with open("data.yaml", "w") as f:
    yaml.dump(data_yaml, f)

print("data.yaml created")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.7 MB/s eta 0:00:0000:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
data.yaml created


In [50]:
!cat data.yaml

names:
- Aluminium foil
- Battery
- Blister pack
- Bottle
- Bottle cap
- Broken glass
- Can
- Carton
- Cigarette
- Cup
- Food waste
- Glass jar
- Lid
- Other plastic
- Paper
- Paper bag
- Plastic bag & wrapper
- Plastic container
- Plastic glooves
- Plastic utensils
- Pop tab
- Rope & strings
- Scrap metal
- Shoe
- Squeezable tube
- Straw
- Styrofoam piece
- Unlabeled litter
nc: 28
path: /kaggle/working/TACO/data
train: .
val: .


In [51]:
!ls TACO/data/batch_1 | head

000000.jpg
000001.jpg
000003.jpg
000004.jpg
000005.jpg
000006.jpg
000007.jpg
000008.jpg
000010.jpg
000011.jpg


In [52]:
import os
import shutil

label_dir = "/kaggle/working/yolo_labels"
image_root = "/kaggle/working/TACO/data"

for file in os.listdir(label_dir):

    if file.endswith(".txt"):

        parts = file.split("_")

        batch_folder = parts[0] + "_" + parts[1]   # batch_13

        image_name = "_".join(parts[2:]).replace(".txt",".jpg")

        dest_folder = os.path.join(image_root, batch_folder)

        label_name = image_name.replace(".jpg",".txt")

        shutil.move(
            os.path.join(label_dir,file),
            os.path.join(dest_folder,label_name)
        )

print("Labels moved successfully")

Labels moved successfully


In [53]:
!ls TACO/data/batch_1 | head

000000.jpg
000000.txt
000001.jpg
000001.txt
000003.jpg
000003.txt
000004.jpg
000004.txt
000005.jpg
000005.txt


In [54]:
!ls yolo_labels | wc -l

0


In [55]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640
)

Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a627177ac00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,   